# 1: Quality control, ambient RNA correction (SoupX), and doublet detection (scDblFinder)

Single-nucleus RNA-seq data have several technical features that strongly influence how preprocessing should be performed. One major issue is the presence of many zeros (“dropouts”), where transcripts that are truly expressed in a nucleus are not detected due to technical limitations of the assay. Another complication is that low-quality nuclei, ambient RNA, and doublets can be mixed with real biological variation, making it easy to accidentally remove true signal or retain poor-quality nuclei if quality control is not chosen carefully.

Because of this, early preprocessing steps such as mt, hb and rb counts removal, ambient RNA correction, and doublet detection have a large impact on downstream clustering, annotation, and differential expression analyses. Overly strict filtering can remove rare but biologically meaningful populations, whereas overly permissive filtering allows low-quality nuclei to blur cluster boundaries and distort gene expression patterns.

In this notebook, we:

- perform nucleus-level quality control on one embryonic mouse brain sample,
- correct ambient RNA contamination using **SoupX**,
- detect likely doublets using **scDblFinder**, and
- save both a QC-filtered object (with doublet labels) and a singlet-only object as AnnData files for downstream analysis.

**Inputs**

- Raw count matrix in CSV format with **genes as rows** and **cells/nuclei as columns**, e.g. `<sample>.csv`. # e.g. 96_6_M_KO_edda.csv

**Outputs**

- `<sample>_filtered_qc_doublets.h5ad`: QC-filtered data with SoupX-corrected counts and doublet annotations.
- `<sample>_filtered_qc_singlets.h5ad`: singlet-only AnnData object.

Run this notebook top-to-bottom after creating the Python + R environment as described in the project README.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import anndata2ri
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from scipy.stats import median_abs_deviation
from rpy2.robjects.conversion import localconverter
from anndata2ri import py2rpy, rpy2py
from rpy2.robjects import pandas2ri, numpy2ri

# Scanpy and plotting defaults
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sns.set(style="whitegrid")

## 1.1. Configuration

All user-editable parameters are collected here for clarity and reproducibility.

In [ ]:
# Input CSV: genes as rows, cells/nuclei as columns
input_csv_file = "<sample>.csv"

# QC thresholds
mt_threshold = 0.05   # maximum fraction of mitochondrial counts
nmads = 5             # MAD multiplier for outlier detection

# Minimum cells per gene after QC
min_cells_per_gene = 10

# Output file names
output_qc_doublets = "<sample>_filtered_qc_doublets.h5ad"
output_qc_singlets = "<sample>_filtered_qc_singlets.h5ad"

## 1.2. Load raw counts and create AnnData

We load the raw count matrix from CSV (genes × cells), transpose it to the common cells × genes convention, and construct an `AnnData` object. Gene names are made unique to avoid downstream issues.

In [ ]:
# Load CSV data: genes as rows, cells as columns
df = sc.read_csv(input_csv_file).T  # transpose → cells x genes

# Create AnnData object
adata = sc.AnnData(df)
adata.var_names_make_unique()
adata

## 1.3. Compute QC metrics

We annotate mitochondrial, ribosomal, and hemoglobin genes and compute per-cell QC metrics using Scanpy. These metrics will be used to filter out low-quality nuclei.

In [ ]:
# Annotate gene sets for QC
adata.var["mt"] = adata.var_names.str.startswith("mt-")
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
adata.var["hb"] = adata.var_names.str.contains("^Hb[^(p)]")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    inplace=True,
    percent_top=[20],
    log1p=True,
)

# Summary statistics before filtering
print("Summary Statistics BEFORE Filtering:")
print(
    adata.obs[
        [
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "pct_counts_ribo",
            "pct_counts_hb",
            "pct_counts_in_top_20_genes",
        ]
    ].describe()
)

## 1.4. Visualize QC metrics before filtering

We inspect the distributions of library size, number of detected genes, mitochondrial fraction, and related QC covariates prior to filtering.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axs[0, 0])
axs[0, 0].set_title("Total Counts Before Filtering")
axs[0, 0].set_xlim(0, 70000)

sns.violinplot(y=adata.obs["pct_counts_mt"], ax=axs[0, 1])
axs[0, 1].set_title("Mitochondrial % Before Filtering")

sns.scatterplot(
    x=adata.obs["total_counts"],
    y=adata.obs["n_genes_by_counts"],
    hue=adata.obs["pct_counts_mt"],
    palette="magma",
    ax=axs[0, 2],
)
axs[0, 2].set_title("Counts vs Genes (Colored by Mito %) Before Filtering")

sns.violinplot(y=adata.obs["pct_counts_ribo"], ax=axs[1, 0])
axs[1, 0].set_title("Ribosomal % Before Filtering")

sns.violinplot(y=adata.obs["pct_counts_hb"], ax=axs[1, 1])
axs[1, 1].set_title("Hemoglobin % Before Filtering")

sns.violinplot(y=adata.obs["pct_counts_in_top_20_genes"], ax=axs[1, 2])
axs[1, 2].set_title("Top 20 Genes % Before Filtering")

plt.tight_layout()
plt.show()

## 1.5. Identify and filter low-quality nuclei

We use a MAD-based rule to flag outliers in library size, number of detected genes, and the fraction of counts in the top 20 genes, and we separately flag nuclei with high mitochondrial fractions. Nuclei flagged by any of these criteria are removed.

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    return (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        M > np.median(M) + nmads * median_abs_deviation(M)
    )

# Mark outliers
adata.obs["outlier"] = (
    is_outlier(adata, "log1p_total_counts", nmads)
    | is_outlier(adata, "log1p_n_genes_by_counts", nmads)
    | is_outlier(adata, "pct_counts_in_top_20_genes", nmads)
)
adata.obs["mt_outlier"] = adata.obs["pct_counts_mt"] > mt_threshold

print(f"Cells flagged as outliers: {adata.obs['outlier'].sum()}")
print(f"Cells flagged as mitochondrial outliers: {adata.obs['mt_outlier'].sum()}")

# Filter low-quality cells
print(f"Cells before filtering: {adata.n_obs}")
adata_filtered = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()
print(f"Cells after filtering: {adata_filtered.n_obs}")

# Summary after filtering
print("\nSummary Statistics AFTER Filtering:")
print(
    adata_filtered.obs[
        [
            "total_counts",
            "n_genes_by_counts",
            "pct_counts_mt",
            "pct_counts_ribo",
            "pct_counts_hb",
            "pct_counts_in_top_20_genes",
        ]
    ].describe()
)

## 1.6. Visualize QC metrics after filtering

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

sns.histplot(adata_filtered.obs["total_counts"], bins=100, kde=False, ax=axs[0, 0])
axs[0, 0].set_title("Total Counts After Filtering")
axs[0, 0].set_xlim(0, 70000)

sns.violinplot(y=adata_filtered.obs["pct_counts_mt"], ax=axs[0, 1])
axs[0, 1].set_title("Mitochondrial % After Filtering")

sns.scatterplot(
    x=adata_filtered.obs["total_counts"],
    y=adata_filtered.obs["n_genes_by_counts"],
    hue=adata_filtered.obs["pct_counts_mt"],
    palette="magma",
    ax=axs[0, 2],
)
axs[0, 2].set_title("Counts vs Genes (Colored by Mito %) After Filtering")

sns.violinplot(y=adata_filtered.obs["pct_counts_ribo"], ax=axs[1, 0])
axs[1, 0].set_title("Ribosomal % After Filtering")

sns.violinplot(y=adata_filtered.obs["pct_counts_hb"], ax=axs[1, 1])
axs[1, 1].set_title("Hemoglobin % After Filtering")

sns.violinplot(y=adata_filtered.obs["pct_counts_in_top_20_genes"], ax=axs[1, 2])
axs[1, 2].set_title("Top 20 Genes % After Filtering")

plt.tight_layout()
plt.show()

## 1.7. Clustering for SoupX ambient RNA correction

SoupX benefits from having coarse clusters of cells to help estimate the ambient RNA profile. Here, we generate a simple Leiden clustering on a normalized/log-transformed copy of the filtered data and store the cluster labels as `soupx_groups`.

In [ ]:
# Preserve the raw counts before further processing 
adata_filtered.layers["raw_counts"] = adata_filtered.X.copy()

# Preprocessing
adata_pp = adata_filtered.copy()
sc.pp.normalize_per_cell(adata_pp)
sc.pp.log1p(adata_pp)
sc.pp.pca(adata_pp)
sc.pp.neighbors(adata_pp)
sc.tl.leiden(adata_pp, key_added="soupx_groups")
sc.tl.umap(adata_pp)

soupx_groups = adata_pp.obs["soupx_groups"]

# UMAP colored by SoupX clusters
sc.pl.umap(adata_pp, color="soupx_groups")

# free memory
del adata_pp

## 1.8. Prepare matrices and run SoupX in R

We now transfer raw counts and cluster labels into R via `rpy2`, set up the SoupX environment, and run ambient RNA correction.

In [ ]:
# Prepare count matrices for SoupX (raw counts)
data_tod = adata_filtered.layers["raw_counts"].T
cells = adata_filtered.obs_names
genes = adata_filtered.var_names
data = adata_filtered.layers["raw_counts"].T

# 1. Load with suppress
ro.r("""
suppressPackageStartupMessages({
  library(SoupX)
  library(Matrix)
})
print("SoupX loaded successfully")
""")

# Pass objects to R
with (ro.default_converter + numpy2ri.converter + pandas2ri.converter + anndata2ri.converter).context():
    ro.globalenv["data"] = data
    ro.globalenv["data_tod"] = data_tod
    ro.globalenv["genes"] = genes
    ro.globalenv["cells"] = cells
    ro.globalenv["soupx_groups"] = soupx_groups

## 1.9. Run SoupX correction in R

In [ ]:
ro.r("""
set.seed(123)
library(SoupX)
library(Matrix)

rownames(data) <- genes
colnames(data) <- cells
data <- as(data, "sparseMatrix")
data_tod <- as(data_tod, "sparseMatrix")

sc <- SoupChannel(data_tod, data, calcSoupProfile = FALSE)

soupProf <- data.frame(row.names = rownames(data),
                       est = rowSums(data) / sum(data),
                       counts = rowSums(data))
sc <- setSoupProfile(sc, soupProf)
sc <- setClusters(sc, soupx_groups)
sc <- autoEstCont(sc, contaminationRange = c(0.01, 0.1), doPlot = TRUE)

cat("Estimated contamination fraction:", sc$conEstimate, "\\n")

out <- adjustCounts(sc, method = "soupOnly", roundToInt = TRUE)
out <- as.matrix(out)
""")

# Get SoupX output from R
out = ro.globalenv["out"]

# Convert R matrix → NumPy (genes x cells → cells x genes)
out_np = np.array(out).T

# Assign to AnnData layers (cells x genes)
adata_filtered.layers["soupX_counts"] = out_np
adata_filtered.layers["raw_counts"] = adata_filtered.X.copy()

# Replace main matrix with SoupX-corrected counts
adata_filtered.X = adata_filtered.layers["soupX_counts"]

## 1.10. Filter lowly expressed genes

We remove genes that are detected in fewer than `min_cells_per_gene` cells. This reduces noise from very rare or artifactual features.

In [ ]:
print(f"Genes before filtering: {adata_filtered.n_vars}")
sc.pp.filter_genes(adata_filtered, min_cells=min_cells_per_gene)
print(f"Genes after filtering: {adata_filtered.n_vars}")

## 1.11. Doublet detection with scDblFinder

We call scDblFinder in R via `rpy2` on the SoupX-corrected count matrix to estimate doublet scores and classify each nucleus as singlet or doublet.

In [ ]:
data_mat = adata_filtered.X.T

with (ro.default_converter + numpy2ri.converter + pandas2ri.converter).context():
    ro.globalenv["data_mat"] = data_mat

ro.r("""
library(scDblFinder)
library(SingleCellExperiment)
library(BiocParallel)

set.seed(123)
sce <- SingleCellExperiment(list(counts = data_mat))
sce <- scDblFinder(sce)

doublet_score <- sce$scDblFinder.score
doublet_class <- sce$scDblFinder.class
""")

adata_filtered.obs["scDblFinder_score"] = np.array(ro.globalenv["doublet_score"])
adata_filtered.obs["scDblFinder_class"] = np.array(ro.globalenv["doublet_class"])  # numeric

print("scDblFinder doublet call counts:")
print(adata_filtered.obs["scDblFinder_class"].value_counts())

## 1.12. Visualize doublets vs singlets on UMAP

In [ ]:
sc.pp.pca(adata_filtered, n_comps=50)
sc.pp.neighbors(adata_filtered, use_rep="X_pca")
sc.tl.umap(adata_filtered)

sc.pl.umap(
    adata_filtered,
    color="scDblFinder_class",
    title="UMAP: Doublets vs Singlets",
)

## 1.13. Save QC outputs and singlet-only dataset

We save the QC-filtered dataset (with doublet annotations) and a singlet-only AnnData object for downstream integration and clustering.

In [ ]:
# Save filtered (with doublets)
adata_filtered.write(output_qc_doublets)

# Map 1→"singlet", 2→"doublet"
adata_filtered.obs["scDblFinder_class_word"] = adata_filtered.obs["scDblFinder_class"].map(
    {1: "singlet", 2: "doublet"}
)

# Extract singlets (numeric 1)
singlets = adata_filtered.obs["scDblFinder_class"] == 1
adata_singlets = adata_filtered[singlets].copy()

# Recompute QC on singlets
sc.pp.calculate_qc_metrics(
    adata_singlets,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True,
)

adata_singlets.obs["scDblFinder_class"] = "singlet"
adata_singlets.write(output_qc_singlets)

print(f"Singlets saved: {adata_singlets.n_obs}")